# 04 · Tracking inventory

**Question.** What does the game actually track, and where is the tracking unreliable?
**Answer.** 66 events, mostly in lower camelCase. Twelve quality issues, the main ones being purchases tracked on Android only, 10.4% of events without session id and a revenue property renamed mid-window. They are written up in `docs/tracking_plan.md`.
**Method.** Everything is observed in the data (`game_events.inventory`); the game's code and design documents were not available, so triggers and uses in the plan are interpretations. Example values are kept only for properties with 20 values or fewer, so that no identifier or receipt is published. Same output as `python -m game_events inventory`.

In [1]:
from game_events import inventory
from game_events.data import connect_events

con, readable, unreadable = connect_events()
events, props, users = inventory.build_inventory(con)      # writes docs/*.csv; the property scan takes a few minutes
print(len(events), "event names,", len(props), "event/property pairs,", len(users), "player properties")

66 event names, 189 event/property pairs, 8 player properties


## 1. Events

In [2]:
events.head(25)

,event_name,events,players,first_day,last_day,pct_without_session
0,tokenCashout,9425816,64318,2025-02-28,2025-04-10,0.0
1,lostCustomer,6638996,58866,2025-02-28,2025-04-10,0.0
2,interstitialLifecycle,5129605,64613,2025-02-28,2025-04-10,0.0
3,rvLifecycle,4460380,65083,2025-02-28,2025-04-10,0.0
4,appBackground,4207157,65316,2025-02-28,2025-04-10,0.0
5,transferMoney,3890433,63927,2025-02-28,2025-04-10,0.0
6,appFocus,3626170,65542,2025-02-28,2025-04-10,0.0
7,achievementUpdated,3525219,59054,2025-02-28,2025-04-10,100.0
8,deviceBroken,3240686,55712,2025-02-28,2025-04-10,0.0
9,deviceRepair,2961672,53868,2025-02-28,2025-04-10,0.0


Events that start after the first days or stop before the end point to tracking changes, or to the 2025-03-27 data gap:

In [3]:
events[(events["first_day"].astype(str) > "2025-03-01") | (events["last_day"].astype(str) < "2025-04-09")]

,event_name,events,players,first_day,last_day,pct_without_session
23,rvRewardClaim,498847,28327,2025-03-12,2025-04-10,0.0
24,_propertiesUpdate,412485,33495,2025-03-24,2025-04-10,0.0
45,_experimentJoin,3379,3329,2025-03-07,2025-04-10,100.0
49,inAppPurchase,411,264,2025-02-28,2025-03-21,0.0
63,gem6,41,28,2025-03-01,2025-04-08,0.0
64,gem5,33,28,2025-03-02,2025-04-10,0.0


## 2. Naming conventions

In [4]:
events["style"] = events["event_name"].map(inventory.naming_style)
(events.groupby("style")["event_name"]
 .agg(count="count", examples=lambda s: ", ".join(s.head(8))))

,count,examples
style,,
camelCase,52,"tokenCashout, lostCustomer, interstitialLifecy..."
contains a digit,9,"noAds24HoursOffer, gem1, gem2, gem3, robot3Day..."
single lowercase word,1,robot
system (leading underscore),4,"_propertiesUpdate, _sessionStart, _identityCre..."


Names with a digit (`gem1`, `gem2`, ...) are product names used as event names: each new product creates a new event, while `purchaseCompleted` already carries a product property.

## 3. Event properties

In [5]:
props.head(25)

,event_name,property,n_events,approx_distinct_values,first_day,last_day,example,pct_of_event
87,_experimentJoin,experiment_id,3379,1,2025-03-07,2025-04-10,544551aa-3fb2-4f7e-bd04-f46ea72777ea,100.0
88,_experimentJoin,experiment_name,3379,1,2025-03-07,2025-04-10,AdsConfig_iOS,100.0
89,_experimentJoin,experiment_phase_id,3379,1,2025-03-07,2025-04-10,7822f0a0-95bd-44fc-8339-9d88026394c1,100.0
90,_experimentJoin,experiment_phase_name,3379,1,2025-03-07,2025-04-10,AdsConfig_iOS,100.0
91,_experimentJoin,experiment_variant_name,3379,3,2025-03-07,2025-04-10,Variant-A,100.0
187,_experimentJoin,experiment_variant_id,3379,3,2025-03-07,2025-04-10,b4a1b997-ce93-4b22-8a33-0e5edf9660f5,100.0
57,_identityCreate,identity_id,51743,45813,2025-02-28,2025-04-10,,100.0
47,_sessionStart,identity_id,286686,68943,2025-02-28,2025-04-10,,100.0
48,_sessionStart,token_expires_at,286686,249430,2025-02-28,2025-04-10,,100.0
151,_sessionStart,session_id,286686,262545,2025-02-28,2025-04-10,,100.0


Properties missing from part of their event's rows, among the frequent events (optional properties, or added during the window):

In [6]:
props[(props["pct_of_event"] < 100) & (props["n_events"] > 1000)].sort_values("pct_of_event").head(25)

,event_name,property,n_events,approx_distinct_values,first_day,last_day,example,pct_of_event
96,adImpression,amount,578369,452948,2025-03-24,2025-04-10,,30.4
146,adImpression,value,1322491,1029119,2025-02-28,2025-04-10,,69.6
49,_sessionStart,refresh_token_expires_at,275178,296873,2025-02-28,2025-04-10,,96.0


## 4. Player properties

`user_properties_custom` is repeated on every event: a seeded 1% sample is enough to list its keys.

In [7]:
users

,property,n_in_sample,distinct_values_in_sample,example,pct_of_events
0,currentMap,707844,7,1,99.8
1,fbInstallationId,706521,54626,,99.6
2,appVersion,706521,19,0.21.20,99.6
3,robotOwned,706510,2,False,99.6
4,noAdsOwned,706510,2,False,99.6
5,adjustId,706093,53990,,99.6
6,dbChurnPrediction,171920,23860,,24.2
7,dbPurchasePrediction,9966,648,,1.4


## Output

`docs/event_inventory.csv`, `docs/event_properties.csv` and `docs/user_properties.csv`, the base of `docs/tracking_plan.md`. A property found on a small share of events cannot be used as a player attribute; very rare keys may be missing from the sample.